# Raw Episode Collection

Collect raw Super Mario Bros. level 1-1 episodes (aiming for around 80 episodes for initial smoke-test)

Artifact structure:

```text
data/raw/episodes/ep_000001/
  frames.mkv     # lossless video, preferably FFV1
  steps.jsonl    # one JSON row per frame/action
  meta.json      # episode-level metadata
```

In [57]:
# Initialize the data collection directory structure
from __future__ import annotations

import sys
import time
import gzip
import json
import random
import shutil
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Any

import numpy as np

try:
    import imageio.v2 as imageio
except ImportError:
    imageio = None

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "data-collection" else Path.cwd()

# Ensure rlab is in python path
RLAB_SRC = REPO_ROOT / "data-collection" / "agent" / "rlab" / "src"
if RLAB_SRC.exists() and str(RLAB_SRC) not in sys.path:
    sys.path.insert(0, str(RLAB_SRC))

from rlab.env import EnvConfig, make_eval_vec_env
from stable_baselines3 import PPO

RAW_EPISODES_DIR = REPO_ROOT / "data" / "raw" / "episodes"

print(f"Raw episodes will be written to: {RAW_EPISODES_DIR}")


Raw episodes will be written to: /Users/soheilchavoshi/Projects/mario-world/data/raw/episodes


In [58]:
@dataclass
class CollectionConfig:
    game: str = "SuperMarioBros-Nes-v0"
    level: str = "1-1"
    fps: int = 60
    uncap_fps: bool = True  # Uncap FPS by default to avoid timer sleep stutters and keep gameplay frame-accurate
    max_steps: int = 512
    random_start_enabled: bool = True
    random_start_min_seconds: float = 0.0
    # Match the 512-step collection horizon: 512 * 4 emulator frames / 60 Hz.
    random_start_max_seconds: float = 23.0
    random_start_policy_steps: int | None = None
    emulator_fps: int = 60
    frame_skip: int = 4
    stop_on_level_change: bool = True
    goal_start_buffer_steps: int = 30
    action_set: str = "simple"
    policy_name: str = "ppo_pretrained"


CONFIG = CollectionConfig()
asdict(CONFIG) #convert to dictionary


{'game': 'SuperMarioBros-Nes-v0',
 'level': '1-1',
 'fps': 60,
 'uncap_fps': True,
 'max_steps': 512,
 'random_start_enabled': True,
 'random_start_min_seconds': 0.0,
 'random_start_max_seconds': 25.0,
 'random_start_policy_steps': None,
 'emulator_fps': 60,
 'frame_skip': 4,
 'stop_on_level_change': True,
 'goal_start_buffer_steps': 30,
 'action_set': 'simple',
 'policy_name': 'ppo_pretrained'}

In [59]:
import queue
import threading

def json_sanitize(obj):
    if isinstance(obj, (np.integer, np.floating, np.bool_)):
        return obj.item()
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {str(k): json_sanitize(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [json_sanitize(v) for v in obj]
    elif isinstance(obj, (int, float, str, bool, type(None))):
        return obj
    else:
        return str(obj)


class EpisodeWriter:
    def __init__(self, episode_dir: Path, config: CollectionConfig, metadata: dict[str, Any] | None = None):
        self.episode_dir = episode_dir
        self.tmp_dir = episode_dir.with_name(episode_dir.name + ".tmp")
        self.config = config
        self.metadata = metadata or {}
        self.q = queue.Queue()
        self.worker = threading.Thread(target=self._write_loop)

    def _write_loop(self):
        with imageio.get_writer(
            self.tmp_dir / "frames.mkv",
            fps=self.config.fps,
            codec="ffv1",
            macro_block_size=None,
        ) as video_writer, (self.tmp_dir / "steps.jsonl").open("w") as steps_file:
            while True:
                item = self.q.get()
                if item is None:
                    break
                frame, row = item
                video_writer.append_data(frame)
                steps_file.write(json.dumps(row, default=json_sanitize) + "\n")

    def __enter__(self):
        #Overwrite pre-existing episodes
        if self.episode_dir.exists():
            shutil.rmtree(self.episode_dir)
        if self.tmp_dir.exists():
            shutil.rmtree(self.tmp_dir)
        self.tmp_dir.mkdir(parents=True)

        metadata = {**asdict(self.config), **self.metadata}
        (self.tmp_dir / "meta.json").write_text(json.dumps(metadata, indent=2, default=json_sanitize) + "\n")
        self.worker.start()
        return self

    def write_step(self, t: int, frame: np.ndarray, action: int, reward: float, done: bool, info: dict[str, Any]):
        row = {
            "t": int(t),
            "action": int(action),
            "reward": float(reward),
            "done": bool(done),
            "info": json_sanitize(info),
        }
        self.q.put((np.asarray(frame, dtype=np.uint8), row))

    def __exit__(self, exc_type, exc, tb):
        self.q.put(None)
        self.worker.join()
        if exc_type is None:
            self.tmp_dir.rename(self.episode_dir)
        else:
            shutil.rmtree(self.tmp_dir, ignore_errors=True)
        return False


In [60]:
def make_env(config: CollectionConfig):
    rlab_config = EnvConfig(
        game=config.game,
        env_provider="supermariobrosnes-turbo",
        action_set=config.action_set,
        frame_skip=config.frame_skip,
        observation_size=84,
        obs_crop=(32, 0, 0, 0),
        max_pool_frames=False,
        max_episode_steps=4500,  # Keep full env max steps to prevent premature VecEnv auto-reset & frame-stack corruption
    )
    return make_eval_vec_env(rlab_config, n_envs=1, seed=random.randint(0, 1000000))


def load_policy(config: CollectionConfig):
    model_path = REPO_ROOT / "data-collection" / "agent" / "NES-SuperMarioBros_Level1-1_gray84-hudcrop-stack4-simple_ppo" / "model.zip"
    if not model_path.exists():
        model_path = REPO_ROOT / "training" / "agent" / "NES-SuperMarioBros_Level1-1_gray84-hudcrop-stack4-simple_ppo" / "model.zip"
    return PPO.load(model_path)


def unwrap_env(env):
    curr = env
    while True:
        if hasattr(curr, "venv"):
            curr = curr.venv
        elif hasattr(curr, "envs") and len(curr.envs) > 0:
            curr = curr.envs[0]
        elif hasattr(curr, "env"):
            curr = curr.env
        else:
            break
    return curr


def get_rgb_frame(env, obs) -> np.ndarray:
    if hasattr(env, "render"):
        frame = env.render(mode="rgb_array") if "mode" in getattr(env.render, "__code__", object()).co_varnames else env.render()
        if frame is not None:
            if isinstance(frame, list) and len(frame) > 0:
                frame = frame[0]
            return np.asarray(frame, dtype=np.uint8)
    return np.asarray(obs, dtype=np.uint8)


def policy_action(policy, obs, deterministic: bool = True, state=None, episode_start=None):
    if hasattr(policy, "predict"):
        action, next_state = policy.predict(
            obs,
            state=state,
            episode_start=episode_start,
            deterministic=deterministic,
        )
        return action, next_state
    action = policy(obs)
    if isinstance(action, tuple):
        return action[0], action[1] if len(action) > 1 else None
    return action, None


def choose_action(env, policy, obs, config: CollectionConfig, state=None, episode_start=None) -> tuple[int, Any]:
    action, next_state = policy_action(policy, obs, deterministic=True, state=state, episode_start=episode_start)
    return int(np.asarray(action).flat[0]), next_state


def step_env(env, action: int):
    step_res = env.step([action] if hasattr(env, "num_envs") else action)
    if len(step_res) == 4:
        obs, reward, done, info_res = step_res
        reward_val = float(reward[0]) if isinstance(reward, (list, np.ndarray)) else float(reward)
        done_val = bool(done[0]) if isinstance(done, (list, np.ndarray)) else bool(done)
        info_dict = info_res[0] if isinstance(info_res, list) and len(info_res) > 0 else info_res
    else:
        obs, reward_val, terminated, truncated, info_dict = step_res
        done_val = bool(terminated or truncated)
    return obs, reward_val, done_val, dict(info_dict or {})


def target_level_id(config: CollectionConfig) -> str:
    world, stage = (int(value) for value in config.level.split("-"))
    return f"{world - 1}-{stage - 1}"


def left_target_level(info: dict[str, Any], config: CollectionConfig) -> bool:
    if not config.stop_on_level_change:
        return False
    if bool(info.get("level_changed", False)):
        return True
    level_id = info.get("level_id")
    return level_id is not None and str(level_id) != target_level_id(config)


def target_level_finished(info: dict[str, Any], config: CollectionConfig) -> bool:
    return (
        left_target_level(info, config)
        or bool(info.get("level_complete", False))
        or bool(info.get("completion_event", False))
    )


def prepare_random_start(env, policy, config: CollectionConfig):
    """Advance the emulator without recording, then begin a new policy episode there."""
    min_steps = round(config.random_start_min_seconds * config.emulator_fps / config.frame_skip)
    max_steps = round(config.random_start_max_seconds * config.emulator_fps / config.frame_skip)
    if min_steps < 0 or max_steps < min_steps:
        raise ValueError("Random-start seconds must satisfy 0 <= min <= max.")

    if not config.random_start_enabled:
        warmup_steps = 0
    elif config.random_start_policy_steps is not None:
        warmup_steps = config.random_start_policy_steps
        if not min_steps <= warmup_steps <= max_steps:
            raise ValueError("random_start_policy_steps is outside the configured range.")
    else:
        warmup_steps = random.SystemRandom().randint(min_steps, max_steps)
    requested_warmup_steps = warmup_steps
    resample_count = 0
    while True:
        obs_res = env.reset()
        obs = obs_res[0] if isinstance(obs_res, tuple) else obs_res
        policy_state = None
        episode_start = np.ones(1, dtype=bool)

        for warmup_t in range(warmup_steps):
            action, policy_state = choose_action(env, policy, obs, config, policy_state, episode_start)
            episode_start = np.zeros(1, dtype=bool)
            obs, _, done, info = step_env(env, action)
            if done or target_level_finished(info, config):
                # The requested start is in the goal sequence or after it. Retry before the goal.
                max_safe_steps = warmup_t - config.goal_start_buffer_steps
                if max_safe_steps < min_steps:
                    raise RuntimeError("Level 1-1 ended before a safe random start was available.")
                warmup_steps = random.SystemRandom().randint(min_steps, min(max_steps, max_safe_steps))
                resample_count += 1
                break
        else:
            break

    # PPO is feed-forward, but this also clears state if a recurrent policy is used later.
    policy_state = None
    episode_start = np.ones(1, dtype=bool)
    actual_seconds = warmup_steps * config.frame_skip / config.emulator_fps
    metadata = {
        "random_start_seconds": actual_seconds,
        "random_start_policy_steps": warmup_steps,
        "random_start_requested_policy_steps": requested_warmup_steps,
        "random_start_resample_count": resample_count,
        "random_start_frame_skip": config.frame_skip,
        "policy_episode_reset_at_random_start": True,
    }
    return obs, policy_state, episode_start, metadata


In [61]:
def run_episode(
    episode_id: int,
    config: CollectionConfig = CONFIG,
) -> dict[str, Any]:
    episode_dir = RAW_EPISODES_DIR / f"ep_{episode_id:06d}"
    env = make_env(config)
    policy = load_policy(config)

    total_reward = 0.0
    recorded_steps = 0
    termination_reason = "max_steps"

    try:
        obs, policy_state, episode_start, start_metadata = prepare_random_start(env, policy, config)

        with EpisodeWriter(episode_dir, config, metadata=start_metadata) as writer:
            for t in range(config.max_steps):
                if not config.uncap_fps and config.fps > 0:
                    time.sleep(1.0 / config.fps)

                action, policy_state = choose_action(env, policy, obs, config, policy_state, episode_start)
                obs, reward_val, done_val, info_dict = step_env(env, action)
                episode_start = np.asarray([done_val], dtype=bool)

                # Do not save the goal sequence, 1-2, or any later level.
                if target_level_finished(info_dict, config):
                    termination_reason = (
                        "level_complete"
                        if bool(info_dict.get("level_complete", False)) or bool(info_dict.get("completion_event", False))
                        else "level_change"
                    )
                    break

                total_reward += float(reward_val)

                frame = get_rgb_frame(env, obs)
                writer.write_step(t, frame, action, reward_val, done_val, info_dict)
                recorded_steps += 1

                if done_val:
                    termination_reason = "environment_done"
                    break
    finally:
        env.close()

    return {
        "episode_id": episode_id,
        "steps": recorded_steps,
        "reward": total_reward,
        "random_start_seconds": start_metadata["random_start_seconds"],
        "termination_reason": termination_reason,
        "dir": str(episode_dir),
    }


## Collection Plan

Collect raw episodes using the pre-trained PPO policy. The collection plan samples starts across the full 34-second, 512-step horizon. It selects one random offset from each equal part of the level, so a small batch covers the start, middle, and end. The warm-up frames are discarded. If the sampled offset is in the goal sequence or after it, the collector resamples before the goal. The PPO policy state is then reset, so the saved episode starts from the sampled game state. The collector stops before it writes the goal sequence or a frame from level 1-2. The sampled offset is saved in `meta.json`.


In [62]:
def make_collection_plan(num_episodes: int) -> list[CollectionConfig]:
    """Create one independent configuration and random start for every episode."""
    if num_episodes < 0:
        raise ValueError("num_episodes must be non-negative.")
    if not CONFIG.random_start_enabled:
        return [replace(CONFIG, random_start_policy_steps=0) for _ in range(num_episodes)]

    min_steps = round(CONFIG.random_start_min_seconds * CONFIG.emulator_fps / CONFIG.frame_skip)
    max_steps = round(CONFIG.random_start_max_seconds * CONFIG.emulator_fps / CONFIG.frame_skip)
    candidate_steps = list(range(min_steps, max_steps + 1))
    if not candidate_steps:
        raise ValueError("Random-start range contains no policy steps.")

    rng = random.SystemRandom()
    warmup_steps = []
    while len(warmup_steps) < num_episodes:
        batch_size = min(num_episodes - len(warmup_steps), len(candidate_steps))
        # Select one random offset from each equal part of the level.
        # A small batch therefore covers the start, middle, and end of the level.
        batch = []
        for index in range(batch_size):
            lower = min_steps + len(candidate_steps) * index // batch_size
            upper = min_steps + len(candidate_steps) * (index + 1) // batch_size - 1
            batch.append(rng.randint(lower, upper))
        rng.shuffle(batch)
        warmup_steps.extend(batch)

    return [
        replace(CONFIG, random_start_policy_steps=steps)
        for steps in warmup_steps[:num_episodes]
    ]


def collect_dataset(num_episodes: int, dry_run: bool = True, overwrite: bool = True) -> list[dict[str, Any]]:
    plan = make_collection_plan(num_episodes)
    summaries = []

    for episode_id, config in enumerate(plan, start=1):
        episode_dir = RAW_EPISODES_DIR / f"ep_{episode_id:06d}"

        if episode_dir.exists() and not overwrite:
            summaries.append({"episode_id": episode_id, "status": "skipped_exists"})
            continue

        if dry_run:
            summaries.append({"episode_id": episode_id, "status": "planned"})
            continue

        summary = run_episode(episode_id, config)
        summary["status"] = "written"
        summaries.append(summary)

    return summaries


In [63]:
# Smoke test once make_env/load_policy are wired.
summary = run_episode(episode_id=1, config=CONFIG)
print("Smoke test episode summary:", summary)

# Verify recorded artifact
ep_dir = Path(summary["dir"])
assert ep_dir.exists(), f"Episode directory {ep_dir} does not exist!"

with imageio.get_reader(ep_dir / "frames.mkv") as video_reader:
    frame_count = sum(1 for _ in video_reader)

with (ep_dir / "steps.jsonl").open() as f:
    step_count = sum(1 for _ in f)

print(f"Recorded frame count: {frame_count}")
print(f"Recorded step count: {step_count}")
assert frame_count == step_count, f"Frame count ({frame_count}) does not match step count ({step_count})"

# Preview the 80-episode collection plan without writing files.
plan_preview = collect_dataset(num_episodes=5, dry_run=False)
print("Plan preview (first 5):", plan_preview[:5])
print("Total episodes planned:", len(plan_preview))

Smoke test episode summary: {'episode_id': 1, 'steps': 394, 'reward': 264.8500087670982, 'random_start_seconds': 3.6, 'termination_reason': 'level_complete', 'dir': '/Users/soheilchavoshi/Projects/mario-world/data/raw/episodes/ep_000001'}
Recorded frame count: 394
Recorded step count: 394
Plan preview (first 5): [{'episode_id': 1, 'steps': 197, 'reward': 40.17500053718686, 'random_start_seconds': 16.733333333333334, 'termination_reason': 'level_complete', 'dir': '/Users/soheilchavoshi/Projects/mario-world/data/raw/episodes/ep_000001', 'status': 'written'}, {'episode_id': 2, 'steps': 238, 'reward': 88.70000230148435, 'random_start_seconds': 14.0, 'termination_reason': 'level_complete', 'dir': '/Users/soheilchavoshi/Projects/mario-world/data/raw/episodes/ep_000002', 'status': 'written'}, {'episode_id': 3, 'steps': 308, 'reward': 171.72500537708402, 'random_start_seconds': 9.333333333333334, 'termination_reason': 'level_complete', 'dir': '/Users/soheilchavoshi/Projects/mario-world/data/ra